In [46]:
import re
import pandas as pd
import numpy as np

MODALITY_PATTERNS = {
    "prohibition": [
        r"\bshall not\b",
        r"\bmust not\b",
        r"\bmay not\b",
        r"\bcannot\b",
        r"\bwill not\b",
        r"\bprohibited\b"
    ],
    "obligation": [
        r"\bshall\b",
        r"\bmust\b",
        r"\bis required to\b",
        r"\bare required to\b",
        r"\bwill\b"
    ],
    "permission": [
        r"\bmay\b",
        r"\bpermitted to\b",
        r"\ballowed to\b",
        r"\bentitled to\b"
    ]
}

def detect_modality(text):
    text_lower = text.lower()

    # Check prohibition first because "shall not"
    # also contains "shall"
    for pattern in MODALITY_PATTERNS["prohibition"]:
        if re.search(pattern, text_lower):
            return "prohibition"

    for pattern in MODALITY_PATTERNS["obligation"]:
        if re.search(pattern, text_lower):
            return "obligation"

    for pattern in MODALITY_PATTERNS["permission"]:
        if re.search(pattern, text_lower):
            return "permission"

    return "none"


# Test
tests = [
    "The Buyer shall make payment within 10 days.",
    "The Buyer shall not disclose Confidential Information.",
    "The Buyer may disclose information to its employees."
]

for t in tests:
    print(t)
    print("→", detect_modality(t))
    print()

The Buyer shall make payment within 10 days.
→ obligation

The Buyer shall not disclose Confidential Information.
→ prohibition

The Buyer may disclose information to its employees.
→ permission



In [47]:
NEGATION_PATTERNS = [
    r"\bnot\b",
    r"\bno\b",
    r"\bnever\b",
    r"\bwithout\b",
    r"\bprohibited\b",
    r"\bforbidden\b"
]

def detect_negation(text):
    text_lower = text.lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in NEGATION_PATTERNS
    )


for t in tests:
    print(t)
    print("Negation:", detect_negation(t))

The Buyer shall make payment within 10 days.
Negation: False
The Buyer shall not disclose Confidential Information.
Negation: True
The Buyer may disclose information to its employees.
Negation: False


In [48]:
TIME_PATTERNS = [
    r"\bwithin\s+\d+\s+(?:day|days|hour|hours|month|months|year|years)\b",
    r"\bafter\s+\d+\s+(?:day|days|month|months|year|years)\b",
    r"\bbefore\s+\d+\s+(?:day|days|month|months|year|years)\b",
    r"\bfor\s+\d+\s+(?:day|days|month|months|year|years)\b",
    r"\buntil\b[^,.;]*",
    r"\bfrom\b[^,.;]*\buntil\b[^,.;]*"
]

def extract_temporal(text):
    matches = []

    for pattern in TIME_PATTERNS:
        matches.extend(
            re.findall(pattern, text, flags=re.IGNORECASE)
        )

    return list(dict.fromkeys(matches))


for t in tests:
    print(t)
    print("Temporal:", extract_temporal(t))

The Buyer shall make payment within 10 days.
Temporal: ['within 10 days']
The Buyer shall not disclose Confidential Information.
Temporal: []
The Buyer may disclose information to its employees.
Temporal: []


In [49]:
NUMBER_PATTERNS = [
    r"\b\d+(?:\.\d+)?\s*%",
    r"\b\d+(?:\.\d+)?\s+(?:day|days|hour|hours|month|months|year|years)\b",
    r"\b(?:USD|EUR|GBP|INR|\$|€|£)\s*\d+(?:,\d{3})*(?:\.\d+)?\b",
    r"\b\d+(?:,\d{3})*(?:\.\d+)?\b"
]

def extract_numbers(text):
    matches = []

    for pattern in NUMBER_PATTERNS:
        matches.extend(
            re.findall(pattern, text, flags=re.IGNORECASE)
        )

    return list(dict.fromkeys(matches))


for t in tests:
    print(t)
    print("Numbers:", extract_numbers(t))

The Buyer shall make payment within 10 days.
Numbers: ['10 days', '10']
The Buyer shall not disclose Confidential Information.
Numbers: []
The Buyer may disclose information to its employees.
Numbers: []


In [50]:
def extract_proposition(text):
    return {
        "text": text,
        "modality": detect_modality(text),
        "negation": detect_negation(text),
        "temporal": extract_temporal(text),
        "numbers": extract_numbers(text)
    }


example_clause = """
The Receiving Party shall not disclose any Confidential Information
to any third party without prior written consent.
"""

proposition = extract_proposition(example_clause)

proposition

{'text': '\nThe Receiving Party shall not disclose any Confidential Information\nto any third party without prior written consent.\n',
 'modality': 'prohibition',
 'negation': True,
 'temporal': [],
 'numbers': []}

In [51]:
structured_clauses = clauses_df.copy()

structured = structured_clauses["text"].apply(
    extract_proposition
)

structured_df = pd.json_normalize(structured)

structured_clauses = pd.concat(
    [
        structured_clauses.reset_index(drop=True),
        structured_df.reset_index(drop=True)
    ],
    axis=1
)

print(structured_clauses.shape)
structured_clauses.head()

(47321, 8)


,document_id,span_id,text,text,modality,negation,temporal,numbers
0,34,0,NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT,NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT,none,False,[],[]
1,34,1,This NON-DISCLOSURE AND CONFIDENTIALITY AGREEM...,This NON-DISCLOSURE AND CONFIDENTIALITY AGREEM...,none,False,[],[]
2,34,2,(i) the Office of the United Nations High Comm...,(i) the Office of the United Nations High Comm...,none,False,[],"[94, 1202]"
3,34,3,"(ii) ________________________ , a company esta...","(ii) ________________________ , a company esta...",none,False,[],[]
4,34,4,________________________ and having its princi...,________________________ and having its princi...,none,False,[],[]


In [52]:
# Recreate extract_parties()

PARTY_PATTERNS = [
    r"\bReceiving Party\b",
    r"\bDisclosing Party\b",
    r"\bBuyer\b",
    r"\bSeller\b",
    r"\bVendor\b",
    r"\bCustomer\b",
    r"\bCompany\b",
    r"\bEmployer\b",
    r"\bEmployee\b",
    r"\bParty\b",
    r"\bParties\b"
]


def extract_parties(text):
    if not isinstance(text, str):
        text = str(text)

    found = []

    for pattern in PARTY_PATTERNS:
        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )
        found.extend(matches)

    return list(dict.fromkeys(
        x.lower() for x in found
    ))


print("extract_parties() recreated successfully.")

extract_parties() recreated successfully.


In [53]:
# FIX structured_clauses completely

# Remove duplicate column names
structured_clauses = structured_clauses.loc[
    :, ~structured_clauses.columns.duplicated()
].copy()

# Check what we have
print("Columns:")
print(structured_clauses.columns.tolist())

print("\nText column type:")
print(type(structured_clauses["text"]))

# If text somehow still isn't a Series, rebuild structured_clauses
if not isinstance(structured_clauses["text"], pd.Series):

    structured_clauses = clauses_df.copy()

    structured_data = []

    for text in structured_clauses["text"].astype(str):
        structured_data.append({
            "modality": detect_modality(text),
            "negation": detect_negation(text),
            "temporal": extract_temporal(text),
            "numbers": extract_numbers(text)
        })

    structured_features = pd.DataFrame(structured_data)

    structured_clauses = pd.concat(
        [
            structured_clauses.reset_index(drop=True),
            structured_features.reset_index(drop=True)
        ],
        axis=1
    )

# Now extract parties
structured_clauses["parties"] = (
    structured_clauses["text"]
    .astype(str)
    .apply(extract_parties)
)

print("\nSUCCESS!")
print(structured_clauses.shape)

display(
    structured_clauses[
        ["document_id", "span_id", "text",
         "modality", "negation", "temporal",
         "numbers", "parties"]
    ].head(10)
)

Columns:
['document_id', 'span_id', 'text', 'modality', 'negation', 'temporal', 'numbers']

Text column type:
<class 'pandas.core.series.Series'>

SUCCESS!
(47321, 8)


,document_id,span_id,text,modality,negation,temporal,numbers,parties
0,34,0,NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT,none,False,[],[],[]
1,34,1,This NON-DISCLOSURE AND CONFIDENTIALITY AGREEM...,none,False,[],[],[]
2,34,2,(i) the Office of the United Nations High Comm...,none,False,[],"[94, 1202]",[]
3,34,3,"(ii) ________________________ , a company esta...",none,False,[],[],[company]
4,34,4,________________________ and having its princi...,none,False,[],[],[]
5,34,5,________________________________________________,none,False,[],[],[]
6,34,6,(hereinafter the “Bidder” or the “Recipient”).,none,False,[],[],[]
7,34,7,The Discloser and Recipient are also referred ...,none,False,[],[],"[party, parties]"
8,34,8,RECITALS,none,False,[],[],[]
9,34,9,"WHEREAS in connection with RFP/2014/620, Reque...",none,False,[],"[2014, 620]",[]


In [54]:
# Cell 29 — Extract conditional and scope information

CONDITION_PATTERNS = [
    r"\bif\b[^,.;]*",
    r"\bunless\b[^,.;]*",
    r"\bprovided that\b[^,.;]*",
    r"\bsubject to\b[^,.;]*",
    r"\bin the event that\b[^,.;]*",
    r"\bwhere\b[^,.;]*",
    r"\bonly if\b[^,.;]*",
    r"\bexcept\b[^,.;]*",
    r"\bexcept that\b[^,.;]*",
    r"\bfor\b[^,.;]*transactions",
    r"\bwith respect to\b[^,.;]*"
]


def extract_conditions(text):
    if not isinstance(text, str):
        text = str(text)

    matches = []

    for pattern in CONDITION_PATTERNS:
        matches.extend(
            re.findall(
                pattern,
                text,
                flags=re.IGNORECASE
            )
        )

    return list(dict.fromkeys(matches))


structured_clauses["conditions"] = (
    structured_clauses["text"]
    .astype(str)
    .apply(extract_conditions)
)

print("Condition extraction completed.")

display(
    structured_clauses[
        ["text", "conditions"]
    ].head(10)
)

Condition extraction completed.


,text,conditions
0,NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT,[]
1,This NON-DISCLOSURE AND CONFIDENTIALITY AGREEM...,[]
2,(i) the Office of the United Nations High Comm...,[]
3,"(ii) ________________________ , a company esta...",[]
4,________________________ and having its princi...,[]
5,________________________________________________,[]
6,(hereinafter the “Bidder” or the “Recipient”).,[]
7,The Discloser and Recipient are also referred ...,[]
8,RECITALS,[]
9,"WHEREAS in connection with RFP/2014/620, Reque...",[]


In [55]:
from itertools import combinations

# Only use clauses that contain meaningful legal language
candidate_clauses = structured_clauses[
    structured_clauses["modality"] != "none"
].copy()

print("Candidate legal clauses:", len(candidate_clauses))

Candidate legal clauses: 15633


In [56]:
def create_clause_pairs(clauses):
    pairs = []

    for document_id, group in clauses.groupby("document_id"):

        records = group.to_dict("records")

        # Avoid creating an enormous number of pairs
        # for very large contracts
        for a, b in combinations(records, 2):

            pairs.append({
                "document_id": document_id,

                "span_a": a["span_id"],
                "text_a": a["text"],
                "modality_a": a["modality"],
                "negation_a": a["negation"],
                "temporal_a": a["temporal"],
                "numbers_a": a["numbers"],
                "parties_a": a["parties"],
                "conditions_a": a["conditions"],

                "span_b": b["span_id"],
                "text_b": b["text"],
                "modality_b": b["modality"],
                "negation_b": b["negation"],
                "temporal_b": b["temporal"],
                "numbers_b": b["numbers"],
                "parties_b": b["parties"],
                "conditions_b": b["conditions"]
            })

    return pd.DataFrame(pairs)


# Start with a sample so we don't accidentally create
# millions of pairs and exhaust Kaggle memory.

sample_docs = candidate_clauses[
    "document_id"
].drop_duplicates().sample(
    min(100, candidate_clauses["document_id"].nunique()),
    random_state=42
)

sample_clauses = candidate_clauses[
    candidate_clauses["document_id"].isin(sample_docs)
].copy()

clause_pairs_sample = create_clause_pairs(
    sample_clauses
)

print("Documents:", sample_docs.nunique())
print("Candidate pairs:", len(clause_pairs_sample))

Documents: 100
Candidate pairs: 42838
